# Pengenalan Bahasa Isyarat Alfabet dari Citra Tangan Menggunakan CNN
**Tim Axcelle** | Mata Kuliah Pembelajaran Mendalam | Semester Ganjil TA. 2026-2027

Notebook ini disusun mengikuti nomor bagian pada format laporan (Bagian A).
Jalankan dari atas ke bawah di Google Colab (runtime GPU: *Runtime > Change runtime type > T4 GPU*).

> **Catatan dataset:** Sign Language MNIST berisi alfabet isyarat **ASL (American Sign Language)**, bukan SIBI.
> Seluruh penamaan di notebook, laporan, dan website harus memakai "ASL".

| Bagian notebook | Bagian laporan |
|---|---|
| 0. Setup | 7. Teknologi |
| 1. Load data | 2. Dataset |
| 2. EDA | 3. EDA |
| 3. Preprocessing | 4. Preprocessing |
| 4-5. Model dan pelatihan | 5. Model |
| 6. Evaluasi | 6. Hasil evaluasi |
| 7. Skenario perbandingan | 6d |
| 8-9. Fitur unik | 5e |
| 10-11. Inferensi dan ekspor | 8a, Deploy |
| 12. Ekspor website | Deploy (Bagian B) |

**Perubahan dari versi sebelumnya**
1. Penamaan SIBI diganti ASL.
2. Skenario dan model utama memakai konfigurasi yang sama (epoch, patience); S5 dan baris lr=1e-3 memakai ulang model utama, sehingga angkanya identik.
3. Grad-CAM: `grad_model` dibuat sekali (jauh lebih cepat).
4. Ekspor TFLite memakai model inferensi tanpa layer augmentasi, dengan cadangan lewat SavedModel.
5. Threshold dipilih dari data validation (bukan test).
6. `RandomContrast` memakai `value_range=(0, 1)`, rotasi diperkecil, `fill_mode="nearest"`.
7. Evaluasi tambahan pada test set resmi penuh (sebelum pembersihan) agar sebanding dengan angka umum.

## 0. Setup dan versi library (Laporan bagian 7)

In [ ]:
# Jalankan sekali di awal. tf_keras dibutuhkan agar model kompatibel dengan TensorFlow.js (website)
!pip install -q tf_keras kagglehub

In [ ]:
import os, sys, glob, hashlib, platform, random, time, json
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # Keras 2 (tf_keras): wajib SEBELUM import tensorflow, agar model bisa diekspor ke TensorFlow.js
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(name):
    """Simpan gambar ke folder outputs agar bisa langsung dipakai di laporan."""
    plt.savefig(f"{OUT_DIR}/{name}.png", dpi=200, bbox_inches="tight")

print("Python      :", platform.python_version())
print("TensorFlow  :", tf.__version__)
import tf_keras
print("Keras       : tf_keras", tf_keras.__version__, "(legacy)")
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU         :", gpus if gpus else "tidak ada (CPU)")

## 1. Load dataset (Laporan bagian 2)
Sumber: https://www.kaggle.com/datasets/datamunge/sign-language-mnist

Jika `kagglehub` gagal, unduh manual `sign_mnist_train.csv` dan `sign_mnist_test.csv` dari Kaggle,
unggah ke Colab (ikon folder di kiri), lalu isi `DATA_DIR` dengan foldernya (misalnya `"/content"`).

In [ ]:
DATA_DIR = None   # isi manual bila kagglehub gagal, contoh: "/content"

if DATA_DIR is None:
    try:
        import kagglehub
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    DATA_DIR = kagglehub.dataset_download("datamunge/sign-language-mnist")
print("Folder data:", DATA_DIR)

def find_csv(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    assert hits, f"{name} tidak ditemukan di {DATA_DIR}"
    return hits[0]

train_df = pd.read_csv(find_csv("sign_mnist_train.csv"))
test_df  = pd.read_csv(find_csv("sign_mnist_test.csv"))

print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df.head()

In [ ]:
# Informasi dataset untuk laporan bagian 2b
pixel_cols = [c for c in train_df.columns if c.startswith("pixel")]
print("Jumlah fitur piksel   :", len(pixel_cols), "(=> 28 x 28, grayscale)")
print("Rentang nilai piksel  :", train_df[pixel_cols].values.min(), "-", train_df[pixel_cols].values.max())
print("Jumlah kelas (train)  :", train_df["label"].nunique())
print("Label unik (train)    :", sorted(train_df["label"].unique()))
print("Label unik (test)     :", sorted(test_df["label"].unique()))
print("Missing value train   :", int(train_df.isna().sum().sum()))
print("Missing value test    :", int(test_df.isna().sum().sum()))
print("Baris duplikat train  :", int(train_df.duplicated().sum()))
print("Baris duplikat test   :", int(test_df.duplicated().sum()))

# Label 9 (J) dan 25 (Z) tidak ada karena keduanya membutuhkan gerakan.
# Pemetaan label asli -> huruf: label l  =>  huruf chr(65 + l)
unique_labels = sorted(train_df["label"].unique())
class_names = [chr(65 + l) for l in unique_labels]
NUM_CLASSES = len(class_names)
label_to_idx = {l: i for i, l in enumerate(unique_labels)}
print("\nKelas akhir:", class_names, "| jumlah =", NUM_CLASSES)

## 2. Exploratory Data Analysis (Laporan bagian 3)

In [ ]:
# 3a. Distribusi kelas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, title in zip(axes, [train_df, test_df], ["Train", "Test"]):
    counts = df["label"].map(lambda l: chr(65 + l)).value_counts().sort_index()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color="steelblue")
    ax.set_title(f"Distribusi kelas - {title}")
    ax.set_xlabel("Huruf"); ax.set_ylabel("Jumlah")
plt.tight_layout(); save_fig("eda_distribusi_kelas"); plt.show()

counts_train = train_df["label"].value_counts()
IMBALANCE_RATIO = counts_train.max() / counts_train.min()
print(f"Rasio kelas terbanyak/tersedikit (train): {IMBALANCE_RATIO:.2f}")

In [ ]:
# 3e. Contoh sampel per kelas
def row_to_img(row):
    return row[pixel_cols].values.reshape(28, 28).astype("uint8")

fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for ax, l in zip(axes.ravel(), unique_labels):
    sample = train_df[train_df["label"] == l].iloc[0]
    ax.imshow(row_to_img(sample), cmap="gray")
    ax.set_title(chr(65 + l)); ax.axis("off")
plt.suptitle("Contoh sampel per kelas", y=1.0)
plt.tight_layout(); save_fig("eda_sampel_per_kelas"); plt.show()

In [ ]:
# Variasi dalam satu kelas (untuk membahas variasi pencahayaan/posisi)
demo_label = unique_labels[0]
subset = train_df[train_df["label"] == demo_label].sample(12, random_state=SEED)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, (_, r) in zip(axes.ravel(), subset.iterrows()):
    ax.imshow(row_to_img(r), cmap="gray"); ax.axis("off")
plt.suptitle(f"Variasi sampel dalam satu kelas (huruf {chr(65+demo_label)})")
plt.tight_layout(); save_fig("eda_variasi_dalam_kelas"); plt.show()

In [ ]:
# Statistik intensitas piksel
mean_img = train_df[pixel_cols].mean().values.reshape(28, 28)
std_img  = train_df[pixel_cols].std().values.reshape(28, 28)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(mean_img, cmap="gray"); axes[0].set_title("Rata-rata piksel")
axes[1].imshow(std_img, cmap="gray");  axes[1].set_title("Standar deviasi piksel")
axes[2].hist(train_df[pixel_cols].values.ravel(), bins=50, color="steelblue")
axes[2].set_title("Histogram intensitas piksel")
for a in axes[:2]: a.axis("off")
plt.tight_layout(); save_fig("eda_statistik_piksel"); plt.show()
print("Rata-rata intensitas:", train_df[pixel_cols].values.mean().round(2),
      "| std:", train_df[pixel_cols].values.std().round(2))

In [ ]:
# Cek kemiripan train vs test (potensi kebocoran data, untuk bagian 1a / 4e / 6e)
def row_hash(arr):
    return hashlib.md5(arr.tobytes()).hexdigest()

train_hash = train_df[pixel_cols].values.astype("uint8")
test_hash  = test_df[pixel_cols].values.astype("uint8")
train_hashes = set(row_hash(r) for r in train_hash)
test_overlap_mask = np.array([row_hash(r) in train_hashes for r in test_hash])
print("Sampel test yang identik persis dengan sampel train:", int(test_overlap_mask.sum()),
      f"({test_overlap_mask.mean()*100:.2f}%)")

**Temuan EDA (isi ulang dengan angka asli di laporan bagian 3f):**
- Distribusi kelas relatif seimbang / tidak seimbang (lihat rasio di atas) -> menentukan perlu atau tidaknya class weight.
- Piksel bernilai 0-255 -> perlu normalisasi ke 0-1.
- Citra hanya 28x28 grayscale -> detail jari terbatas, huruf yang bentuknya mirip berpotensi tertukar.
- Variasi dalam kelas (posisi, kecerahan, latar) -> alasan memakai augmentasi.
- Label 9 dan 25 tidak ada -> label perlu dipetakan ulang ke 0..23.

## 3. Preprocessing (Laporan bagian 4)
Urutan: hapus duplikat -> (opsional) bersihkan overlap train/test -> pisah train/val -> normalisasi -> reshape -> one-hot.
Normalisasi hanya memakai pembagian 255 (tidak ada statistik yang dipelajari dari data), sehingga tidak ada kebocoran statistik.

Test set dievaluasi dua kali: **test bersih** (tanpa duplikat dan overlap dengan train; angka utama) dan
**test resmi penuh** (apa adanya dari Kaggle; untuk pembanding dengan angka di literatur).

In [ ]:
REMOVE_TEST_OVERLAP = True   # buang sampel test yang identik dengan train

def prepare(df):
    X = df[pixel_cols].values.astype("float32") / 255.0
    X = X.reshape(-1, 28, 28, 1)
    y = df["label"].map(label_to_idx).values.astype("int32")
    return X, y

# Test resmi penuh (untuk pembanding)
X_test_full, y_test_full = prepare(test_df)

# 4a. Pembersihan duplikat
n_train_before = len(train_df)
train_clean = train_df.drop_duplicates().reset_index(drop=True)
print(f"Train: {n_train_before} -> {len(train_clean)} (duplikat dibuang: {n_train_before - len(train_clean)})")

n_test_before = len(test_df)
test_clean = test_df.drop_duplicates().reset_index(drop=True)
n_test_dup_removed = n_test_before - len(test_clean)
n_test_overlap_removed = 0
if REMOVE_TEST_OVERLAP:
    t_hashes = set(row_hash(r) for r in train_clean[pixel_cols].values.astype("uint8"))
    keep = np.array([row_hash(r) not in t_hashes for r in test_clean[pixel_cols].values.astype("uint8")])
    n_test_overlap_removed = int((~keep).sum())
    test_clean = test_clean[keep].reset_index(drop=True)
print(f"Test: {n_test_before} -> {len(test_clean)} "
      f"(duplikat: {n_test_dup_removed}, overlap dengan train: {n_test_overlap_removed})")

# 4b. Pemisahan fitur/label, remap label, normalisasi, reshape
X_all, y_all = prepare(train_clean)
X_test, y_test = prepare(test_clean)

# 4e. Pembagian data: train / validation (stratified) / test (bawaan dataset)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
total = len(X_train) + len(X_val) + len(X_test)
print(f"Proporsi -> train {len(X_train)/total:.1%} | val {len(X_val)/total:.1%} | test {len(X_test)/total:.1%}")

# One-hot untuk categorical cross-entropy
y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val, NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test, NUM_CLASSES)
y_test_full_oh = keras.utils.to_categorical(y_test_full, NUM_CLASSES)

> **Catatan untuk laporan (4e / 6e):** dataset ini dibuat dari sejumlah kecil foto asli yang diperbanyak dengan augmentasi.
> Pemisahan acak train/val tidak bisa menjamin varian dari foto asli yang sama tidak muncul di kedua sisi,
> sehingga akurasi validasi/test cenderung optimistis. Tuliskan keterbatasan ini secara jujur.

In [ ]:
# 4d. Class weight (dipakai hanya jika distribusi tidak seimbang)
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = {i: w for i, w in enumerate(cw)}
print("Rentang class weight:", round(cw.min(), 3), "-", round(cw.max(), 3))
USE_CLASS_WEIGHT = False   # ubah True jika EDA menunjukkan ketidakseimbangan yang berarti

## 4. Arsitektur model (Laporan bagian 5a-5d)
CNN dibangun dengan Functional API. Augmentasi berada di dalam model, jadi hanya aktif saat training dan otomatis nonaktif saat inferensi.
Tidak memakai flip horizontal karena membalik tangan dapat mengubah makna isyarat.
`fill_mode="nearest"` dipilih karena latar citra bukan hitam murni.

In [ ]:
def make_contrast(factor=0.15):
    # Keras 3: value_range default (0,255); data kita 0-1.
    try:
        return layers.RandomContrast(factor, value_range=(0, 1), name="aug_contrast")
    except TypeError:
        return layers.RandomContrast(factor, name="aug_contrast")

def build_cnn(use_aug=True, use_bn=True, dropout=0.4, filters=(32, 64, 128),
              dense_units=128, rotation=0.05, name="cnn_asl"):
    inputs = keras.Input(shape=(28, 28, 1), name="input")
    x = inputs

    if use_aug:
        x = layers.RandomRotation(rotation, fill_mode="nearest", name="aug_rotation")(x)   # 0.05 = +-18 derajat
        x = layers.RandomZoom(0.10, fill_mode="nearest", name="aug_zoom")(x)
        x = layers.RandomTranslation(0.10, 0.10, fill_mode="nearest", name="aug_translation")(x)
        x = make_contrast(0.15)(x)

    for i, f in enumerate(filters, start=1):
        x = layers.Conv2D(f, 3, padding="same", use_bias=not use_bn, name=f"conv{i}")(x)
        if use_bn:
            x = layers.BatchNormalization(name=f"bn{i}")(x)
        # nama layer aktivasi terakhir dipakai oleh Grad-CAM
        x = layers.Activation("relu", name="last_conv" if i == len(filters) else f"relu{i}")(x)
        if i < len(filters):
            x = layers.MaxPooling2D(2, name=f"pool{i}")(x)
            if dropout:
                x = layers.Dropout(dropout * 0.5, name=f"drop_conv{i}")(x)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(dense_units, activation="relu", name="dense")(x)
    if dropout:
        x = layers.Dropout(dropout, name="drop_dense")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    return keras.Model(inputs, outputs, name=name)

model = build_cnn()
model.summary()
print("Total parameter:", f"{model.count_params():,}")

In [ ]:
try:
    keras.utils.plot_model(model, to_file=f"{OUT_DIR}/arsitektur_model.png", show_shapes=True, dpi=120)
    from IPython.display import Image, display
    display(Image(f"{OUT_DIR}/arsitektur_model.png"))
except Exception as e:
    print("plot_model tidak tersedia (butuh pydot/graphviz). Gunakan model.summary() sebagai gantinya.\n", e)

## 5. Konfigurasi dan pelatihan (Laporan bagian 5c)

In [ ]:
CONFIG = dict(optimizer="Adam", learning_rate=1e-3, batch_size=128, epochs=40,
              loss="categorical_crossentropy", early_stopping_patience=8, reduce_lr_patience=3)
print(json.dumps(CONFIG, indent=2))

def compile_model(m, lr=CONFIG["learning_rate"]):
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def get_callbacks(ckpt_path=None, patience=CONFIG["early_stopping_patience"]):
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=CONFIG["reduce_lr_patience"], min_lr=1e-5),
    ]
    if ckpt_path:
        cbs.append(keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True))
    return cbs

In [ ]:
keras.utils.set_random_seed(SEED)
model = compile_model(build_cnn())
t0 = time.time()
history = model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    batch_size=CONFIG["batch_size"], epochs=CONFIG["epochs"],
    callbacks=get_callbacks(f"{OUT_DIR}/best_model.keras"),
    class_weight=class_weight if USE_CLASS_WEIGHT else None,
    verbose=2)
train_time = time.time() - t0
print(f"Waktu pelatihan: {train_time/60:.1f} menit | epoch berjalan: {len(history.history['loss'])}")

## 6. Evaluasi (Laporan bagian 6)

In [ ]:
# 6c. Grafik loss dan akurasi
def plot_history(h, name="kurva_pelatihan"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(h.history["loss"], label="Train"); axes[0].plot(h.history["val_loss"], label="Validation")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(h.history["accuracy"], label="Train"); axes[1].plot(h.history["val_accuracy"], label="Validation")
    axes[1].set_title("Akurasi"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout(); save_fig(name); plt.show()

plot_history(history)

In [ ]:
# 6b. Metrik pada data test (test bersih = angka utama, test resmi penuh = pembanding)
test_loss, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)
full_loss, full_acc = model.evaluate(X_test_full, y_test_full_oh, verbose=0)
val_loss, val_acc   = model.evaluate(X_val, y_val_oh, verbose=0)
train_loss, train_acc = model.evaluate(X_train, y_train_oh, verbose=0)
print(f"Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f} | Test (bersih) acc: {test_acc:.4f} | Test (resmi penuh) acc: {full_acc:.4f}")
print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Test (bersih) loss: {test_loss:.4f}")
print(f"Selisih train-test akurasi: {(train_acc-test_acc)*100:.2f} poin (indikasi overfitting jika besar)")

LABELS = np.arange(NUM_CLASSES)
probs_test = model.predict(X_test, verbose=0)
y_pred = probs_test.argmax(axis=1)

report = classification_report(y_test, y_pred, labels=LABELS, target_names=class_names,
                               digits=4, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(f"{OUT_DIR}/classification_report.csv")
print(classification_report(y_test, y_pred, labels=LABELS, target_names=class_names, digits=4, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=LABELS)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Prediksi"); plt.ylabel("Aktual"); plt.title("Confusion Matrix (data test bersih)")
save_fig("confusion_matrix"); plt.show()

# 6e. Pasangan huruf yang paling sering tertukar
pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            pairs.append((class_names[i], class_names[j], int(cm[i, j])))
pairs_df = pd.DataFrame(pairs, columns=["Aktual", "Diprediksi", "Jumlah"]).sort_values("Jumlah", ascending=False)
print("Kesalahan terbanyak:")
print(pairs_df.head(10).to_string(index=False))

In [ ]:
# Contoh sampel yang salah diprediksi
wrong_idx = np.where(y_pred != y_test)[0]
print("Jumlah salah prediksi:", len(wrong_idx), "dari", len(y_test))
if len(wrong_idx):
    sel = wrong_idx[:12]
    fig, axes = plt.subplots(2, 6, figsize=(13, 5))
    for ax, k in zip(axes.ravel(), sel):
        ax.imshow(X_test[k, :, :, 0], cmap="gray")
        ax.set_title(f"A:{class_names[y_test[k]]}  P:{class_names[y_pred[k]]}\n({probs_test[k].max():.2f})", fontsize=9)
        ax.axis("off")
    plt.tight_layout(); save_fig("contoh_salah_prediksi"); plt.show()

## 7. Perbandingan skenario (Laporan bagian 6d)
Semua skenario memakai data, seed, epoch maksimum (`CONFIG["epochs"]`), dan patience yang **sama dengan model utama**.
Skenario S5 (lengkap) dan baris lr=1e-3 memakai ulang model utama yang sudah dilatih, sehingga angkanya identik dengan Bagian 6.
Jalankan 2-3 seed (`SEEDS`) jika waktu memungkinkan dan laporkan rata-rata.

In [ ]:
SCENARIOS = {
    "S1 Baseline (tanpa aug, BN, dropout)": dict(use_aug=False, use_bn=False, dropout=0.0),
    "S2 + BatchNorm":                       dict(use_aug=False, use_bn=True,  dropout=0.0),
    "S3 + BatchNorm + Dropout":             dict(use_aug=False, use_bn=True,  dropout=0.4),
    "S4 + Augmentasi saja":                 dict(use_aug=True,  use_bn=False, dropout=0.0),
    "S5 Lengkap (Aug + BN + Dropout)":      dict(use_aug=True,  use_bn=True,  dropout=0.4),
}
SEEDS = [SEED]   # contoh: [42, 7, 123] untuk 3 kali ulang (S5 hanya memakai ulang model utama untuk seed 42)

def eval_model(m):
    tl, ta = m.evaluate(X_test, y_test_oh, verbose=0)
    _, tra = m.evaluate(X_train, y_train_oh, verbose=0)
    return ta, tl, tra - ta

def train_variant(kw, seed, lr=CONFIG["learning_rate"]):
    keras.utils.set_random_seed(seed)
    m = compile_model(build_cnn(**kw), lr=lr)
    m.fit(X_train, y_train_oh, validation_data=(X_val, y_val_oh),
          batch_size=CONFIG["batch_size"], epochs=CONFIG["epochs"],
          callbacks=get_callbacks(), verbose=0)
    return m

rows = []
for name, kw in SCENARIOS.items():
    accs, losses, gaps = [], [], []
    n_params = None
    for s in SEEDS:
        if name.startswith("S5") and s == SEED:
            m = model                      # pakai ulang model utama
        else:
            m = train_variant(kw, s)
        ta, tl, gap = eval_model(m)
        accs.append(ta); losses.append(tl); gaps.append(gap)
        n_params = m.count_params()
    rows.append({"Skenario": name, "Parameter": n_params,
                 "Test Acc": np.mean(accs), "Test Loss": np.mean(losses),
                 "Gap Train-Test": np.mean(gaps)})
    print(f"{name:42s} acc={np.mean(accs):.4f} loss={np.mean(losses):.4f}")

keras.utils.set_random_seed(SEED)
scen_df = pd.DataFrame(rows)
scen_df.to_csv(f"{OUT_DIR}/perbandingan_skenario.csv", index=False)
scen_df

In [ ]:
# Variasi hyperparameter (learning rate) pada model lengkap
lr_rows = []
for lr in [1e-2, 1e-3, 1e-4]:
    if lr == CONFIG["learning_rate"]:
        m = model                          # pakai ulang model utama
    else:
        m = train_variant(dict(), SEED, lr=lr)
    ta, tl, _ = eval_model(m)
    lr_rows.append({"Learning rate": lr, "Test Acc": ta, "Test Loss": tl})
    print(f"lr={lr:g}  acc={ta:.4f}  loss={tl:.4f}")
lr_df = pd.DataFrame(lr_rows)
lr_df.to_csv(f"{OUT_DIR}/perbandingan_lr.csv", index=False)
lr_df

## 7b. Model inferensi (tanpa layer augmentasi)
Untuk deploy dan ekspor TFLite, dipakai model yang identik tetapi tanpa layer augmentasi
(bobot disalin berdasarkan nama layer). Prediksinya diverifikasi sama dengan model terlatih.

In [ ]:
infer_model = build_cnn(use_aug=False, name="cnn_asl_infer")
for layer in infer_model.layers:
    if layer.weights:
        layer.set_weights(model.get_layer(layer.name).get_weights())

p_a = model.predict(X_test[:500], verbose=0)
p_b = infer_model.predict(X_test[:500], verbose=0)
print("Selisih maksimum prediksi model vs model inferensi:", float(np.abs(p_a - p_b).max()))
assert np.abs(p_a - p_b).max() < 1e-4, "Bobot tidak tersalin dengan benar"
print("Parameter model inferensi:", f"{infer_model.count_params():,}")

## 8. Fitur unik 1: Confidence threshold dan deteksi input di luar domain (Laporan bagian 5e)
**Apa:** model menolak prediksi jika confidence (probabilitas softmax tertinggi) di bawah ambang.
**Mengapa:** softmax selalu menghasilkan satu huruf, bahkan untuk foto yang bukan tangan atau kualitasnya buruk.
**Bukti:** bandingkan distribusi confidence data in-domain dengan data di luar domain (Fashion-MNIST dan noise acak).

Threshold **dipilih dari data validation** (bukan test). Tabel pada data test hanya dipakai untuk pelaporan.

In [ ]:
# Data out-of-domain: Fashion-MNIST (28x28 grayscale, objek bukan tangan) dan noise
(_, _), (fx_test, _) = keras.datasets.fashion_mnist.load_data()
N_OOD = 2000
ood_fashion = (fx_test[:N_OOD].astype("float32") / 255.0).reshape(-1, 28, 28, 1)
ood_noise   = rng.uniform(0, 1, size=(N_OOD, 28, 28, 1)).astype("float32")

probs_val = model.predict(X_val, verbose=0)
conf_val, correct_val = probs_val.max(axis=1), (probs_val.argmax(axis=1) == y_val)
conf_in      = probs_test.max(axis=1)
correct_test = (y_pred == y_test)
conf_fashion = model.predict(ood_fashion, verbose=0).max(axis=1)
conf_noise   = model.predict(ood_noise, verbose=0).max(axis=1)

plt.figure(figsize=(9, 4.5))
plt.hist(conf_in, bins=40, alpha=0.6, label="In-domain (test)", density=True)
plt.hist(conf_fashion, bins=40, alpha=0.6, label="OOD: Fashion-MNIST", density=True)
plt.hist(conf_noise, bins=40, alpha=0.6, label="OOD: noise acak", density=True)
plt.xlabel("Confidence (max softmax)"); plt.ylabel("Kepadatan"); plt.legend()
plt.title("Distribusi confidence: in-domain vs out-of-domain")
save_fig("fitur1_distribusi_confidence"); plt.show()

In [ ]:
THRESHOLDS = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]

def threshold_table(conf, correct):
    out = []
    for thr in THRESHOLDS:
        mask = conf >= thr
        out.append({
            "Threshold": thr,
            "Coverage in-domain": mask.mean(),
            "Akurasi yang diterima": correct[mask].mean() if mask.any() else np.nan,
            "OOD Fashion ditolak": (conf_fashion < thr).mean(),
            "OOD noise ditolak": (conf_noise < thr).mean(),
        })
    return pd.DataFrame(out)

thr_val  = threshold_table(conf_val, correct_val)    # dasar pemilihan threshold
thr_test = threshold_table(conf_in, correct_test)    # untuk pelaporan
print("== Tabel VALIDATION (dasar pemilihan) ==")
print(thr_val.round(4).to_string(index=False))

# Aturan pemilihan: akurasi yang diterima >= TARGET_ACC dan coverage >= MIN_COVERAGE,
# lalu ambil threshold dengan penolakan OOD (rata-rata Fashion & noise) tertinggi.
TARGET_ACC, MIN_COVERAGE = 0.99, 0.80
elig = thr_val[(thr_val["Akurasi yang diterima"] >= TARGET_ACC) &
               (thr_val["Coverage in-domain"] >= MIN_COVERAGE)].copy()
if len(elig):
    elig["ood"] = (elig["OOD Fashion ditolak"] + elig["OOD noise ditolak"]) / 2
    CONF_THRESHOLD = float(elig.sort_values(["ood", "Threshold"], ascending=[False, True]).iloc[0]["Threshold"])
else:
    CONF_THRESHOLD = 0.90   # cadangan jika tidak ada threshold yang memenuhi aturan
print(f"\nCONF_THRESHOLD terpilih (dari validation): {CONF_THRESHOLD}")

print("\n== Tabel TEST (pelaporan) ==")
thr_test.to_csv(f"{OUT_DIR}/fitur1_threshold_tradeoff_test.csv", index=False)
thr_val.to_csv(f"{OUT_DIR}/fitur1_threshold_tradeoff_val.csv", index=False)
print(thr_test.round(4).to_string(index=False))

**Catatan:** Fashion-MNIST dan noise hanyalah *proxy* untuk input di luar domain, dan softmax CNN sering terlalu percaya diri
pada input aneh. Laporkan hasilnya apa adanya. Jika memungkinkan, tambahkan foto non-tangan atau foto tangan dari kamera sendiri
(setelah diubah ke 28x28 grayscale) sebagai uji tambahan.

## 9. Fitur unik 2: Grad-CAM (Laporan bagian 5e)
**Apa:** peta panas yang menunjukkan area citra yang paling memengaruhi prediksi.
**Mengapa:** membuktikan model memperhatikan bentuk tangan, bukan latar atau artefak, serta membantu menganalisis kesalahan.
**Bukti:** visualisasi (benar vs salah) dan uji oklusi: menutup area panas harus menurunkan akurasi lebih besar daripada menutup area acak.

In [ ]:
# grad_model dibuat SEKALI (sebelumnya dibuat ulang di setiap pemanggilan sehingga lambat)
grad_model = keras.Model(model.inputs, [model.get_layer("last_conv").output, model.output])

def make_gradcam(img, pred_index=None):
    x = tf.convert_to_tensor(img[np.newaxis], dtype=tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x, training=False)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        score = preds[:, pred_index]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.reduce_sum(conv_out * weights[:, None, None, :], axis=-1)[0]
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (28, 28))[..., 0].numpy()
    return cam, pred_index, preds.numpy()[0]

def show_gradcam(indices, title, fname):
    fig, axes = plt.subplots(2, len(indices), figsize=(2.2 * len(indices), 5))
    for c, k in enumerate(indices):
        cam, p, pr = make_gradcam(X_test[k])
        axes[0, c].imshow(X_test[k, :, :, 0], cmap="gray"); axes[0, c].axis("off")
        axes[0, c].set_title(f"A:{class_names[y_test[k]]} P:{class_names[p]}", fontsize=9)
        axes[1, c].imshow(X_test[k, :, :, 0], cmap="gray")
        axes[1, c].imshow(cam, cmap="jet", alpha=0.45); axes[1, c].axis("off")
        axes[1, c].set_title(f"{pr[p]:.2f}", fontsize=9)
    plt.suptitle(title); plt.tight_layout(); save_fig(fname); plt.show()

right_idx = np.where(y_pred == y_test)[0][:8]
show_gradcam(right_idx, "Grad-CAM: prediksi benar", "fitur2_gradcam_benar")
if len(wrong_idx):
    show_gradcam(wrong_idx[:8], "Grad-CAM: prediksi salah", "fitur2_gradcam_salah")

In [ ]:
# Uji oklusi: tutup 20% piksel terpanas vs 20% piksel acak, ukur penurunan akurasi
N_EVAL = min(1000, len(X_test))
eval_idx = rng.choice(len(X_test), N_EVAL, replace=False)
frac = 0.20
n_mask = int(frac * 28 * 28)

X_cam, X_rand = X_test[eval_idx].copy(), X_test[eval_idx].copy()
for n, k in enumerate(eval_idx):
    cam, _, _ = make_gradcam(X_test[k])
    top = np.argsort(cam.ravel())[::-1][:n_mask]
    flat = X_cam[n, :, :, 0].ravel(); flat[top] = 0.0
    X_cam[n, :, :, 0] = flat.reshape(28, 28)
    rnd = rng.choice(28 * 28, n_mask, replace=False)
    flat = X_rand[n, :, :, 0].ravel(); flat[rnd] = 0.0
    X_rand[n, :, :, 0] = flat.reshape(28, 28)

acc_orig = accuracy_score(y_test[eval_idx], model.predict(X_test[eval_idx], verbose=0).argmax(1))
acc_cam  = accuracy_score(y_test[eval_idx], model.predict(X_cam, verbose=0).argmax(1))
acc_rand = accuracy_score(y_test[eval_idx], model.predict(X_rand, verbose=0).argmax(1))
occ_df = pd.DataFrame({"Kondisi": ["Citra asli", "Area Grad-CAM ditutup (20%)", "Area acak ditutup (20%)"],
                       "Akurasi": [acc_orig, acc_cam, acc_rand]})
occ_df.to_csv(f"{OUT_DIR}/fitur2_uji_oklusi.csv", index=False)
occ_df

## 10. Ekspor model dan fungsi inferensi untuk website (Laporan bagian 8a dan deploy)

In [ ]:
import subprocess, sys, shutil

# 1) Simpan model Keras dan nama kelas (cepat, selalu dilakukan)
infer_model.save(f"{OUT_DIR}/model_asl_cnn.keras")
model.save(f"{OUT_DIR}/model_asl_cnn_dengan_aug.keras")
size_keras = os.path.getsize(f"{OUT_DIR}/model_asl_cnn.keras") / 1024
with open(f"{OUT_DIR}/class_names.json", "w") as f:
    json.dump(class_names, f)
print(f"Ukuran .keras: {size_keras:.1f} KB")

# 2) TFLite: dijalankan di proses terpisah dengan batas waktu 5 menit agar tidak menggantung
tflite_bytes, size_tflite, acc_tfl = None, None, None
sm_dir = os.path.abspath(f"{OUT_DIR}/saved_model_asl")
tflite_path = os.path.abspath(f"{OUT_DIR}/model_asl_cnn.tflite")
try:
    shutil.rmtree(sm_dir, ignore_errors=True)
    tf.saved_model.save(infer_model, sm_dir)
    code = (
        "import tensorflow as tf\n"
        f"c = tf.lite.TFLiteConverter.from_saved_model({sm_dir!r})\n"
        "c.optimizations = [tf.lite.Optimize.DEFAULT]\n"
        f"open({tflite_path!r}, 'wb').write(c.convert())\n"
    )
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print("Konversi TFLite gagal:", r.stderr[-400:])
    else:
        tflite_bytes = open(tflite_path, "rb").read()
except subprocess.TimeoutExpired:
    print("Konversi TFLite dibatalkan (lebih dari 5 menit). Jangan klaim TFLite di laporan.")
except Exception as e:
    print("Konversi TFLite gagal:", repr(e)[:300], "-> jangan klaim TFLite di laporan.")

# 3) Verifikasi akurasi TFLite pada seluruh test bersih
if tflite_bytes is not None:
    size_tflite = len(tflite_bytes) / 1024
    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    inp, outp = interp.get_input_details()[0], interp.get_output_details()[0]
    preds_tfl = []
    t0 = time.time()
    for k in range(len(X_test)):
        interp.set_tensor(inp["index"], X_test[k:k+1])
        interp.invoke()
        preds_tfl.append(interp.get_tensor(outp["index"])[0].argmax())
    tfl_time = time.time() - t0
    acc_tfl = float(np.mean(np.array(preds_tfl) == y_test))
    print(f"Ukuran .tflite: {size_tflite:.1f} KB")
    print(f"Akurasi TFLite pada test bersih: {acc_tfl:.4f} (Keras: {test_acc:.4f}) | waktu {tfl_time:.1f} dtk")
else:
    print("TFLite dilewati. Website tidak membutuhkannya, lanjut ke sel berikutnya.")

In [ ]:
from PIL import Image, ImageOps

def preprocess_photo(path_or_file, crop_frac=1.0):
    """Foto tangan -> grayscale, autocontrast, crop persegi di tengah, resize 28x28, normalisasi 0-1.
    crop_frac < 1.0 memperkecil area crop (mis. 0.8) bila tangan memenuhi bagian tengah foto.
    Di website, tampilkan kotak panduan agar tangan mengisi frame (citra dataset adalah potongan ketat di sekitar tangan)."""
    img = Image.open(path_or_file).convert("L")
    img = ImageOps.autocontrast(img)
    w, h = img.size
    s = int(min(w, h) * crop_frac)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)).resize((28, 28))
    return np.asarray(img, dtype="float32")[None, :, :, None] / 255.0

def predict_letter(path_or_file, mdl=None, threshold=None, topk=3, crop_frac=1.0):
    mdl = mdl or infer_model
    threshold = CONF_THRESHOLD if threshold is None else threshold
    x = preprocess_photo(path_or_file, crop_frac=crop_frac)
    p = mdl.predict(x, verbose=0)[0]
    order = p.argsort()[::-1][:topk]
    result = {"top": [(class_names[i], float(p[i])) for i in order],
              "diterima": bool(p.max() >= threshold)}
    result["huruf"] = class_names[order[0]] if result["diterima"] else None
    return result

# Contoh: uji dengan salah satu citra test yang disimpan sebagai PNG
Image.fromarray((X_test[0, :, :, 0] * 255).astype("uint8")).save("contoh.png")
print(predict_letter("contoh.png"), "| label asli:", class_names[y_test[0]])

## 11. Rangkuman angka untuk laporan dan unduh hasil

In [ ]:
summary = {
    "train_samples": int(len(X_train)), "val_samples": int(len(X_val)),
    "test_samples_bersih": int(len(X_test)), "test_samples_resmi_penuh": int(len(X_test_full)),
    "test_dibuang_duplikat": int(n_test_dup_removed), "test_dibuang_overlap_train": int(n_test_overlap_removed),
    "num_classes": NUM_CLASSES, "total_params": int(model.count_params()),
    "train_acc": float(train_acc), "val_acc": float(val_acc),
    "test_acc_bersih": float(test_acc), "test_acc_resmi_penuh": float(full_acc),
    "test_macro_f1": float(report["macro avg"]["f1-score"]),
    "epochs_run": len(history.history["loss"]), "train_minutes": round(train_time / 60, 2),
    "confidence_threshold": CONF_THRESHOLD,
    "ukuran_keras_kb": round(size_keras, 1),
    "ukuran_tflite_kb": round(size_tflite, 1) if size_tflite else None,
    "akurasi_tflite_test": acc_tfl,
    "tf_version": tf.__version__, "python": platform.python_version(),
}
with open(f"{OUT_DIR}/ringkasan_angka.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 12. Ekspor model dan hasil training untuk website (Laporan bagian 8a dan Bagian B)
Website memakai **model yang sama** dengan notebook ini (`infer_model`, tanpa layer augmentasi), sehingga arsitektur dan angka evaluasi di laporan, notebook, dan website konsisten.

Hasilnya masuk ke `outputs/web/`:
- `model/` (model.json + .bin) → salin ke `public/model/` di project website
- `training-results.ts` → timpa `src/data/training-results.ts`
- `model-config.json` → catatan urutan kelas dan threshold confidence (fitur unik 1)

In [ ]:
import shutil

WEB_DIR = f"{OUT_DIR}/web"
shutil.rmtree(WEB_DIR, ignore_errors=True)
os.makedirs(f"{WEB_DIR}/model")

# Urutan kelas harus sama dengan CLASSES di src/lib/alphabet.ts (A-Y tanpa J)
assert "".join(class_names) == "ABCDEFGHIKLMNOPQRSTUVWXY", "Urutan kelas tidak cocok dengan website"

def export_tfjs(m, out_dir):
    """Tulis format TensorFlow.js (layers-model) langsung dari model Keras 2, tanpa paket tensorflowjs."""
    specs, chunks = [], []
    for w in m.weights:
        arr = np.asarray(w.numpy(), dtype="<f4")
        specs.append({"name": w.name.split(":")[0], "shape": list(arr.shape), "dtype": "float32"})
        chunks.append(arr.tobytes())
    bin_name = "group1-shard1of1.bin"
    with open(os.path.join(out_dir, bin_name), "wb") as f:
        f.write(b"".join(chunks))
    cfg = json.loads(m.to_json())
    model_json = {
        "format": "layers-model",
        "generatedBy": "keras v" + str(cfg.get("keras_version", "2")),
        "convertedBy": "notebook manual exporter",
        "modelTopology": {
            "keras_version": cfg.get("keras_version", "2"),
            "backend": cfg.get("backend", "tensorflow"),
            "model_config": {"class_name": cfg["class_name"], "config": cfg["config"]},
        },
        "weightsManifest": [{"paths": [bin_name], "weights": specs}],
    }
    with open(os.path.join(out_dir, "model.json"), "w") as f:
        json.dump(model_json, f)
    return specs

specs = export_tfjs(infer_model, f"{WEB_DIR}/model")
size = os.path.getsize(f"{WEB_DIR}/model/group1-shard1of1.bin")
expected = sum(int(np.prod(w.shape)) for w in infer_model.weights) * 4
print("Berkas model web:", sorted(os.listdir(f"{WEB_DIR}/model")))
print(f"Ukuran bobot: {size:,} byte (seharusnya {expected:,}) | jumlah tensor: {len(specs)}")
assert size == expected, "Ukuran bobot tidak cocok"

# Hasil training -> src/data/training-results.ts (angka utama = test bersih, sama dengan Bagian 6)
h = history.history
n_ep = len(h["loss"])
results = {
    "testAccuracy": round(float(test_acc), 4),
    "testLoss": round(float(test_loss), 4),
    "valAccuracy": round(float(val_acc), 4),
    "epochs": n_ep,
    "history": [
        {"epoch": i + 1,
         "acc": round(float(h["accuracy"][i]), 4), "valAcc": round(float(h["val_accuracy"][i]), 4),
         "loss": round(float(h["loss"][i]), 4),    "valLoss": round(float(h["val_loss"][i]), 4)}
        for i in range(n_ep)
    ],
}
ts = """/**
 * Dihasilkan otomatis oleh notebook (bagian 12). Timpa file ini setelah menjalankan ulang notebook.
 */
export type TrainingResults = {
  testAccuracy: number // 0-1
  testLoss: number
  valAccuracy: number // 0-1
  epochs: number
  history: { epoch: number; acc: number; valAcc: number; loss: number; valLoss: number }[]
}

export const TRAINING_RESULTS: TrainingResults | null = """ + json.dumps(results, indent=2) + "\n"
with open(f"{WEB_DIR}/training-results.ts", "w") as f:
    f.write(ts)

with open(f"{WEB_DIR}/model-config.json", "w") as f:
    json.dump({"classes": class_names, "inputShape": [28, 28, 1],
               "confidenceThreshold": CONF_THRESHOLD}, f, indent=2)

print(f"training-results.ts ditulis | test acc (bersih) = {test_acc:.4f} | epoch = {n_ep} | threshold = {CONF_THRESHOLD}")

In [ ]:
# Kompres semua keluaran lalu unduh (khusus Colab)
import shutil
shutil.make_archive("hasil_axcelle", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("hasil_axcelle.zip")
except ImportError:
    print("Bukan Colab: ambil berkas dari folder outputs/ dan hasil_axcelle.zip")